**Asignatura:** Programación para Analítica de Datos  
**Serie:** Python desde cero · Cuaderno 2 de 3  
**Modalidad:** trabajo autónomo en Google Colab  
**Duración estimada:** 2 horas y 30 minutos a 3 horas  
**Prerrequisito recomendado:** Cuaderno 1

---

**_Propósito_**

Al terminar podrá:

1. formular comparaciones y expresiones lógicas;
2. combinar condiciones con `and`, `or` y `not`;
3. organizar valores en listas y registros en diccionarios;
4. traducir reglas a `if`, `elif` y `else`;
5. recorrer datos con `for`;
6. controlar repeticiones con `while`;
7. construir funciones con parámetros y retorno;
8. resumir una lista de registros antes de usar `pandas`;
9. verificar cada ejercicio mediante autoevaluación.


## Antes de comenzar

1. Guarde una copia en Drive.
2. Ejecute las celdas en orden.
3. Edite únicamente `# TODO`.
4. Después de cada ejercicio, ejecute la autoevaluación.
5. Antes de programar, identifique **entrada, proceso y salida**.

Convenciones: ✅ aprobado · 🟡 pendiente · 🧩 pista · 🛟 solución · 📊 progreso.


# 0. Prepare el entorno


In [1]:
import csv
import html
import re
from pathlib import Path
from IPython.display import display, HTML

PUNTAJES = {'E1': 8, 'E2': 8, 'E3': 10, 'E4': 10, 'E5': 10, 'E6': 10, 'E7': 10, 'E8': 10, 'E9': 12, 'E10': 12}

if "RESULTADOS" not in globals():
    RESULTADOS = {}
if "INTENTOS" not in globals():
    INTENTOS = {}
if "nombre_estudiante" not in globals():
    nombre_estudiante = ""

def _tarjeta(mensaje, color_fondo, color_borde):
    display(HTML(
        f'''
        <div style="padding:14px 18px;margin:10px 0;border-radius:10px;
                    background:{color_fondo};border-left:6px solid {color_borde};">
            {mensaje}
        </div>
        '''
    ))

def evaluar(ejercicio, prueba, pistas, solucion):
    '''Ejecuta una prueba y entrega apoyo progresivo sin bloquear el cuaderno.'''
    try:
        aprobado = bool(prueba())
        detalle_error = ""
    except Exception as exc:
        aprobado = False
        detalle_error = f"{type(exc).__name__}: {exc}"

    if aprobado:
        RESULTADOS[ejercicio] = PUNTAJES[ejercicio]
        _tarjeta(
            f"<b>✅ {ejercicio} aprobado.</b> Obtuvo {PUNTAJES[ejercicio]} puntos.",
            "#eaf7ef", "#2e8b57"
        )
        return True

    INTENTOS[ejercicio] = INTENTOS.get(ejercicio, 0) + 1
    intento = INTENTOS[ejercicio]
    RESULTADOS.setdefault(ejercicio, 0)

    mensaje = f"<b>🟡 {ejercicio}: todavía no es correcto.</b><br>"
    if detalle_error:
        mensaje += (
            "<span style='color:#8a3b12;'>Detalle técnico: "
            f"{html.escape(detalle_error)}</span><br>"
        )

    if intento == 1:
        mensaje += f"<b>🧩 Pista 1:</b> {html.escape(pistas[0])}"
    elif intento == 2:
        mensaje += f"<b>🧩 Pista 2:</b> {html.escape(pistas[1])}"
    else:
        mensaje += (
            "<b>🛟 Solución de emergencia:</b><br>"
            "<pre style='background:#fff;padding:10px;border-radius:6px;'>"
            f"{html.escape(solucion)}</pre>"
            "Copie la solución en la celda del ejercicio, ejecútela y vuelva a comprobar."
        )

    _tarjeta(mensaje, "#fff7e6", "#f0a000")
    return False

def mostrar_progreso():
    total = sum(RESULTADOS.get(e, 0) for e in PUNTAJES)
    aprobados = [e for e in PUNTAJES if RESULTADOS.get(e, 0) > 0]
    pendientes = [e for e in PUNTAJES if RESULTADOS.get(e, 0) == 0]
    color = "#2e8b57" if total >= 80 else "#f0a000" if total >= 60 else "#b23a48"

    _tarjeta(
        f"<b>📊 Progreso: {total}/100 puntos</b><br>"
        f"Aprobados: {', '.join(aprobados) if aprobados else 'ninguno'}<br>"
        f"Pendientes: {', '.join(pendientes) if pendientes else 'ninguno'}",
        "#f4f7fb", color
    )

def exportar_autoevaluacion(nombre="estudiante"):
    nombre = nombre.strip()
    if not nombre or nombre == "Escriba aquí su nombre":
        nombre = "estudiante"

    nombre_seguro = re.sub(r"[^A-Za-z0-9_-]+", "_", nombre).strip("_") or "estudiante"
    registros = [
        {
            "estudiante": nombre,
            "ejercicio": e,
            "puntaje_obtenido": RESULTADOS.get(e, 0),
            "puntaje_maximo": maximo,
            "intentos": INTENTOS.get(e, 0),
        }
        for e, maximo in PUNTAJES.items()
    ]

    archivo = Path(f"autoevaluacion_cuaderno2_{nombre_seguro}.csv")
    with archivo.open("w", newline="", encoding="utf-8-sig") as salida:
        escritor = csv.DictWriter(salida, fieldnames=list(registros[0].keys()))
        escritor.writeheader()
        escritor.writerows(registros)

    filas = "".join(
        "<tr>"
        f"<td>{html.escape(str(r['ejercicio']))}</td>"
        f"<td>{r['puntaje_obtenido']}</td>"
        f"<td>{r['puntaje_maximo']}</td>"
        f"<td>{r['intentos']}</td>"
        "</tr>"
        for r in registros
    )
    display(HTML(
        "<table style='border-collapse:collapse;'>"
        "<tr><th style='padding:6px;border:1px solid #ccc;'>Ejercicio</th>"
        "<th style='padding:6px;border:1px solid #ccc;'>Obtenido</th>"
        "<th style='padding:6px;border:1px solid #ccc;'>Máximo</th>"
        "<th style='padding:6px;border:1px solid #ccc;'>Intentos</th></tr>"
        f"{filas}</table>"
    ))
    print(f"Archivo creado: {archivo.name}")

    try:
        from google.colab import files
        files.download(str(archivo))
    except Exception:
        print("Fuera de Colab, el archivo quedó guardado en el directorio actual.")

print("✅ Entorno preparado. Puede comenzar.")
mostrar_progreso()


✅ Entorno preparado. Puede comenzar.


## Identificación


In [2]:
# TODO: reemplace el texto por su nombre
nombre_estudiante = "Laura Muñoz"
print("Estudiante:", nombre_estudiante)


Estudiante: Laura Muñoz


# 1. Expresiones lógicas y comparaciones

Una comparación produce `True` o `False`.


## Ejercicio E1 — Evaluar indicadores (8 puntos)

Con ventas de `$450.000` y meta de `$400.000`, calcule:

- `cumple_meta`;
- `supera_medio_millon`;
- `coincide_meta`.


In [4]:
# TODO E1
ventas_dia = 450000
meta_dia = 400000

cumple_meta = ventas_dia >= meta_dia
supera_medio_millon = ventas_dia > 500000
coincide_meta = ventas_dia == meta_dia

print(cumple_meta)
print(supera_medio_millon)
print(coincide_meta)


True
False
False


In [5]:
# Autoevaluación E1
evaluar(
    "E1",
    lambda: (
        cumple_meta is True
        and supera_medio_millon is False
        and coincide_meta is False
    ),
    pistas=[
        "Use >=, > y == según la pregunta.",
        "Compare ventas_dia con meta_dia, 500000 y meta_dia.",
    ],
    solucion='''cumple_meta = ventas_dia >= meta_dia
supera_medio_millon = ventas_dia > 500000
coincide_meta = ventas_dia == meta_dia'''
)


True

# 2. Conectores lógicos

- `and`: todas las condiciones;
- `or`: al menos una;
- `not`: invierte el valor.


## Ejercicio E2 — Combinar reglas (8 puntos)

Cliente activo, saldo `$0` y beneficio de fidelización.

Calcule `puede_comprar`, `requiere_revision` y `aplica_beneficio`.


In [6]:
# TODO E2
cliente_activo = True
saldo_cliente = 0
tiene_beneficio = True

puede_comprar = cliente_activo and saldo_cliente >= 0
requiere_revision = (not cliente_activo) or saldo_cliente < 0
aplica_beneficio = puede_comprar and tiene_beneficio

print(puede_comprar)
print(requiere_revision)
print(aplica_beneficio)


True
False
True


In [7]:
# Autoevaluación E2
evaluar(
    "E2",
    lambda: (
        puede_comprar is True
        and requiere_revision is False
        and aplica_beneficio is True
    ),
    pistas=[
        "Use and para requisitos simultáneos y or para alternativas.",
        "La revisión es (not cliente_activo) or saldo_cliente < 0.",
    ],
    solucion='''puede_comprar = cliente_activo and saldo_cliente >= 0
requiere_revision = (not cliente_activo) or saldo_cliente < 0
aplica_beneficio = puede_comprar and tiene_beneficio'''
)


True

# 3. Listas

Una lista usa corchetes. El primer índice es `0`; `len()` cuenta; `sum()` suma; `max()` obtiene el mayor.


## Ejercicio E3 — Crear y consultar una lista (10 puntos)

Cree `productos` con `"Bowl Andino"`, `"Pasta Urbana"` y `"Ensalada de la Casa"`.

Obtenga `primer_producto`, `ultimo_producto` y `cantidad_productos`.


In [8]:
# TODO E3
productos = [
    "Bowl Andino",
    "Pasta Urbana",
    "Ensalada de la Casa",
]
primer_producto = productos[0]
ultimo_producto = productos[-1]
cantidad_productos = len(productos)

print(primer_producto)
print(ultimo_producto)
print(cantidad_productos)


Bowl Andino
Ensalada de la Casa
3


In [9]:
# Autoevaluación E3
evaluar(
    "E3",
    lambda: (
        productos == ["Bowl Andino", "Pasta Urbana", "Ensalada de la Casa"]
        and primer_producto == "Bowl Andino"
        and ultimo_producto == "Ensalada de la Casa"
        and cantidad_productos == 3
    ),
    pistas=[
        "Las listas usan corchetes y conservan el orden.",
        "Use [0], [-1] y len(productos).",
    ],
    solucion='''productos = [
    "Bowl Andino",
    "Pasta Urbana",
    "Ensalada de la Casa",
]
primer_producto = productos[0]
ultimo_producto = productos[-1]
cantidad_productos = len(productos)'''
)


True

## Ejercicio E4 — Resumir una lista (10 puntos)

Para `[320000, 450000, 280000, 510000]`, calcule total, promedio y máximo.


In [10]:
# TODO E4
ventas = [320000, 450000, 280000, 510000]

total_ventas = sum(ventas)
promedio_ventas = total_ventas / len(ventas)
venta_maxima = max(ventas)

print(total_ventas)
print(promedio_ventas)
print(venta_maxima)


1560000
390000.0
510000


In [11]:
# Autoevaluación E4
evaluar(
    "E4",
    lambda: (
        total_ventas == 1560000
        and promedio_ventas == 390000
        and venta_maxima == 510000
    ),
    pistas=[
        "Use sum(ventas), len(ventas) y max(ventas).",
        "El promedio es total_ventas / len(ventas).",
    ],
    solucion='''total_ventas = sum(ventas)
promedio_ventas = total_ventas / len(ventas)
venta_maxima = max(ventas)'''
)


True

# 4. Diccionarios

Un diccionario representa un registro mediante pares `clave: valor`.


## Ejercicio E5 — Representar una venta (10 puntos)

Cree un diccionario con cliente 101, producto `"Bowl Andino"`, cantidad 2, precio 24000 y costo 14000.

Calcule ingreso y margen.


In [12]:
# TODO E5
venta = {
    "cliente_id": 101,
    "producto": "Bowl Andino",
    "cantidad": 2,
    "precio_unitario": 24000,
    "costo_unitario": 14000,
}
ingreso_venta = venta["cantidad"] * venta["precio_unitario"]
margen_venta = venta["cantidad"] * (
    venta["precio_unitario"] - venta["costo_unitario"]
)

print(venta)
print(ingreso_venta)
print(margen_venta)


{'cliente_id': 101, 'producto': 'Bowl Andino', 'cantidad': 2, 'precio_unitario': 24000, 'costo_unitario': 14000}
48000
20000


In [13]:
# Autoevaluación E5
evaluar(
    "E5",
    lambda: (
        isinstance(venta, dict)
        and venta["cliente_id"] == 101
        and venta["producto"] == "Bowl Andino"
        and ingreso_venta == 48000
        and margen_venta == 20000
    ),
    pistas=[
        "Use llaves y pares clave: valor.",
        "Acceda a cada dato con venta['clave'].",
    ],
    solucion='''venta = {
    "cliente_id": 101,
    "producto": "Bowl Andino",
    "cantidad": 2,
    "precio_unitario": 24000,
    "costo_unitario": 14000,
}
ingreso_venta = venta["cantidad"] * venta["precio_unitario"]
margen_venta = venta["cantidad"] * (
    venta["precio_unitario"] - venta["costo_unitario"]
)'''
)


True

# 5. Condiciones

`if`, `elif` y `else` traducen reglas de decisión. La sangría define los bloques.


## Ejercicio E6 — Clasificar inventario (10 puntos)

Complete `clasificar_inventario(unidades)`:

- menos de 10: `"Crítico"`;
- de 10 a 24: `"Bajo"`;
- 25 o más: `"Adecuado"`.


In [15]:
# TODO E6
def clasificar_inventario(unidades):
    if unidades < 10:
        return "Crítico"
    elif unidades < 25:
        return "Bajo"
    else:
        return "Adecuado"

print(clasificar_inventario(5))
print(clasificar_inventario(15))
print(clasificar_inventario(30))


Crítico
Bajo
Adecuado


In [16]:
# Autoevaluación E6
evaluar(
    "E6",
    lambda: (
        clasificar_inventario(0) == "Crítico"
        and clasificar_inventario(9) == "Crítico"
        and clasificar_inventario(10) == "Bajo"
        and clasificar_inventario(24) == "Bajo"
        and clasificar_inventario(25) == "Adecuado"
    ),
    pistas=[
        "Use if, elif y else con return.",
        "Evalúe primero unidades < 10 y después unidades < 25.",
    ],
    solucion='''def clasificar_inventario(unidades):
    if unidades < 10:
        return "Crítico"
    elif unidades < 25:
        return "Bajo"
    else:
        return "Adecuado"'''
)


True

# 6. Ciclo `for`

`for` recorre cada elemento de una colección. Se usa para acumular, contar o aplicar reglas.


## Ejercicio E7 — Acumular y contar (10 puntos)

Con ventas `[320000, 450000, 280000, 510000, 390000]` y meta `$400.000`, calcule total y días sobre meta.


In [17]:
# TODO E7
ventas_diarias = [320000, 450000, 280000, 510000, 390000]
meta_diaria = 400000

total_semana = 0
dias_sobre_meta = 0

for venta_actual in ventas_diarias:
    total_semana += venta_actual
    if venta_actual >= meta_diaria:
        dias_sobre_meta += 1

print(total_semana)
print(dias_sobre_meta)


1950000
2


In [18]:
# Autoevaluación E7
evaluar(
    "E7",
    lambda: total_semana == 1950000 and dias_sobre_meta == 2,
    pistas=[
        "Sume cada venta al acumulador.",
        "Si venta_actual >= meta_diaria, aumente el contador.",
    ],
    solucion='''total_semana = 0
dias_sobre_meta = 0

for venta_actual in ventas_diarias:
    total_semana += venta_actual
    if venta_actual >= meta_diaria:
        dias_sobre_meta += 1'''
)


True

# 7. Ciclo `while`

`while` repite mientras una condición sea verdadera. Alguna variable debe cambiar dentro del ciclo para evitar una repetición infinita.


## Ejercicio E8 — Alcanzar una meta (10 puntos)

Se ahorran `$75.000` diarios hasta alcanzar `$300.000`. Calcule saldo y días mediante `while`.


In [20]:
# TODO E8
ahorro_diario = 75000
meta_ahorro = 300000

saldo_acumulado = 0
dias_necesarios = 0

while saldo_acumulado < meta_ahorro:
    saldo_acumulado += ahorro_diario
    dias_necesarios += 1

print(saldo_acumulado)
print(dias_necesarios)


300000
4


In [21]:
# Autoevaluación E8
evaluar(
    "E8",
    lambda: saldo_acumulado == 300000 and dias_necesarios == 4,
    pistas=[
        "Repita mientras saldo_acumulado < meta_ahorro.",
        "Aumente el saldo y el contador dentro del ciclo.",
    ],
    solucion='''saldo_acumulado = 0
dias_necesarios = 0

while saldo_acumulado < meta_ahorro:
    saldo_acumulado += ahorro_diario
    dias_necesarios += 1'''
)


True

# 8. Funciones

Una función agrupa una regla, recibe parámetros y retorna resultados. Evita repetir código y facilita las pruebas.


## Ejercicio E9 — Calcular ingreso neto (12 puntos)

Complete `calcular_ingreso_neto(cantidad, precio_unitario, descuento=0)`.


In [22]:
# TODO E9
def calcular_ingreso_neto(cantidad, precio_unitario, descuento=0):
    ingreso_bruto = cantidad * precio_unitario
    valor_descuento = ingreso_bruto * descuento
    return ingreso_bruto - valor_descuento

print(calcular_ingreso_neto(2, 24000, 0.10))
print(calcular_ingreso_neto(1, 30000))


43200.0
30000


In [23]:
# Autoevaluación E9
evaluar(
    "E9",
    lambda: (
        calcular_ingreso_neto(2, 24000, 0.10) == 43200
        and calcular_ingreso_neto(1, 30000) == 30000
        and calcular_ingreso_neto(3, 10000, 0.20) == 24000
    ),
    pistas=[
        "Calcule ingreso bruto y descuento.",
        "Finalice con return ingreso_bruto - valor_descuento.",
    ],
    solucion='''def calcular_ingreso_neto(cantidad, precio_unitario, descuento=0):
    ingreso_bruto = cantidad * precio_unitario
    valor_descuento = ingreso_bruto * descuento
    return ingreso_bruto - valor_descuento'''
)


True

# 9. Desafío integrador: lista de diccionarios

Esta estructura es el puente hacia el DataFrame del Cuaderno 3.


In [24]:
# Datos autocontenidos
pedidos = [
    {"producto":"Bowl Andino","cantidad":2,"precio_unitario":24000,"costo_unitario":14000},
    {"producto":"Pasta Urbana","cantidad":1,"precio_unitario":28000,"costo_unitario":17000},
    {"producto":"Ensalada de la Casa","cantidad":3,"precio_unitario":22000,"costo_unitario":12000},
]


## Ejercicio E10 — Resumir pedidos (12 puntos)

Complete `resumir_pedidos(registros)` para retornar:

- `numero_pedidos`;
- `unidades_vendidas`;
- `total_ingresos`;
- `total_costos`;
- `margen_bruto`.


In [25]:
# TODO E10
def resumir_pedidos(registros):
    unidades_vendidas = 0
    total_ingresos = 0
    total_costos = 0

    for pedido in registros:
        unidades_vendidas += pedido["cantidad"]
        total_ingresos += pedido["cantidad"] * pedido["precio_unitario"]
        total_costos += pedido["cantidad"] * pedido["costo_unitario"]

    return {
        "numero_pedidos": len(registros),
        "unidades_vendidas": unidades_vendidas,
        "total_ingresos": total_ingresos,
        "total_costos": total_costos,
        "margen_bruto": total_ingresos - total_costos,
    }

resumen = resumir_pedidos(pedidos)
print(resumen)


{'numero_pedidos': 3, 'unidades_vendidas': 6, 'total_ingresos': 142000, 'total_costos': 81000, 'margen_bruto': 61000}


In [26]:
# Autoevaluación E10
def _comprobar_resumen():
    r = resumir_pedidos(pedidos)
    return (
        isinstance(r, dict)
        and r["numero_pedidos"] == 3
        and r["unidades_vendidas"] == 6
        and r["total_ingresos"] == 142000
        and r["total_costos"] == 81000
        and r["margen_bruto"] == 61000
    )

evaluar(
    "E10",
    _comprobar_resumen,
    pistas=[
        "Inicialice acumuladores y recorra cada pedido.",
        "Ingreso y costo son cantidad por valor unitario.",
    ],
    solucion='''def resumir_pedidos(registros):
    unidades_vendidas = 0
    total_ingresos = 0
    total_costos = 0

    for pedido in registros:
        unidades_vendidas += pedido["cantidad"]
        total_ingresos += pedido["cantidad"] * pedido["precio_unitario"]
        total_costos += pedido["cantidad"] * pedido["costo_unitario"]

    return {
        "numero_pedidos": len(registros),
        "unidades_vendidas": unidades_vendidas,
        "total_ingresos": total_ingresos,
        "total_costos": total_costos,
        "margen_bruto": total_ingresos - total_costos,
    }'''
)


True

# 10. Interpretación profesional

1. ¿Qué diferencia conceptual existe entre una lista y un diccionario?
2. ¿Cuándo es preferible `while` frente a `for`?
3. ¿Por qué conviene convertir una regla de negocio en función?
4. ¿Qué información adicional necesitaría para interpretar el margen de `$61.000`?


# 11. Resultado final y exportación

- **80 o más:** continúe al taller de Python aplicado a Data Analytics;
- **60 a 79:** revise condiciones, ciclos y funciones;
- **menos de 60:** repita listas, diccionarios y flujo de control.


In [27]:
mostrar_progreso()


In [28]:
# Exporte el reporte únicamente cuando termine.
exportar_autoevaluacion(nombre_estudiante)


Ejercicio,Obtenido,Máximo,Intentos
E1,8,8,0
E2,8,8,0
E3,10,10,0
E4,10,10,0
E5,10,10,0
E6,10,10,0
E7,10,10,0
E8,10,10,0
E9,12,12,0
E10,12,12,0


Archivo creado: autoevaluacion_cuaderno2_Laura_Mu_oz.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# 12. Lista de comprobación

- [ ] guardé una copia en Drive;
- [ ] ejecuté todas las celdas en orden;
- [ ] obtuve al menos 80/100;
- [ ] respondí la interpretación profesional;
- [ ] generé el CSV final;
- [ ] puedo explicar una condición, un ciclo y una función;
- [ ] conservé el enlace del notebook.

# Recursos oficiales

- [Control de flujo](https://docs.python.org/es/3/tutorial/controlflow.html)
- [Listas](https://docs.python.org/es/3/tutorial/datastructures.html#more-on-lists)
- [Diccionarios](https://docs.python.org/es/3/tutorial/datastructures.html#dictionaries)
- [Funciones](https://docs.python.org/es/3/tutorial/controlflow.html#defining-functions)

> Una solución programable surge cuando los datos, las decisiones y las repeticiones quedan expresados sin ambigüedad.
